# **Unigrams: Full Workflow**
The primary purpose of the unigram workflow is to generate a vocabulary **whitelist** that can be used to filter out unwanted words from a multigram corpus. The workflow consists of two steps: (1) downloading the unigram corpus into a database and (2) filtering and normalizing the corpus and generating the whitelist.

## **Setup**
### Imports

In [5]:
%load_ext autoreload
%autoreload 2

from ngramprep.ngram_filter import FilterConfig, PipelineConfig, load_stopwords
from ngramprep.ngram_filter.lemmatizer import CachedSpacyLemmatizer
from ngramprep.ngram_acquire import download_and_ingest_to_rocksdb
from ngramprep.ngram_filter.pipeline.orchestrator import build_processed_db
from ngramprep.utilities.peek import db_head, db_peek, db_peek_prefix

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### Configure
Here we set basic parameters: the corpus to download, the size of the ngrams to download, and the size of the year bins.

In [4]:
db_path_stub = '/scratch/edk202/NLP_corpora/Google_Books/'
archive_path_stub = None
release = '20200217'
language = 'eng'
ngram_size = 1
bin_size = 1

## **Step 1: Download and Ingest**

In [ ]:
download_and_ingest_to_rocksdb(
    ngram_size=ngram_size,
    repo_release_id=release,
    repo_corpus_id=language,
    db_path_stub=db_path_stub,
    archive_path_stub=archive_path_stub,
    ngram_type="tagged",
    overwrite_db=True,
    open_type="write:packed24",
    compact_after_ingest=True
)

N-GRAM ACQUISITION PIPELINE
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Start Time: 2026-02-07 18:09:49

Download Configuration
════════════════════════════════════════════════════════════════════════════════════════════════════
Ngram repo:           https://books.storage.googleapis.com/?prefix=ngrams/books/20200217/eng/1-
DB path:              /scratch/edk202/NLP_corpora/Google_Books/20200217/eng/1gram_files/1grams.db
File range:           0 to 23
Total files:          24
Files to get:         24
Skipping:             0
Download workers:     24
Batch size:           50,000
Ngram size:           1
Ngram type:           tagged
Overwrite DB:         True
DB Profile:           write:packed24

Download Progress
════════════════════════════════════════════════════════════════════════════════════════════════════


Files Processed:   0%|                                                              | 0/24 [00:00<?]

 ## **Step 2: Filter, Normalize, and Generate Whitelist**
`config.py` contains generic defaults for the filtering pipeline. You can override these defaults by passing option dictionaries to the `build_processed_db` function, as seen below. By default, we:
1. case-normalize the tokens
2. remove tokens containing non-alphanumeric text
3. remove stopwords using the `stop-words` package
4. lemmatize the tokens using the `spaCy` package—first using a lookup table and falling back to rules when lookups fail
5. Create a whitelist of the top 20,000 most frequent words that pass a `pyenchant` spell-check and appear in all corpora from 1900–2019 (inclusive).

In [6]:
stop_set, stop_lang = load_stopwords("en")

filter_config = FilterConfig(
    stop_set=stop_set,
    stop_words_language=stop_lang,
    lemma_gen=CachedSpacyLemmatizer(language="en"),
    ascii_alpha_only=True,
    min_context_tokens=1,
    min_len=3,
    bin_size=bin_size
)

pipeline_config = PipelineConfig(
    # Path construction
    ngram_size=ngram_size,
    repo_release_id=release,
    repo_corpus_id=language,
    db_path_stub=db_path_stub,
    # Pipeline options
    mode="resume",
    num_workers=20,
    num_initial_work_units=300,
    cache_partitions=True,
    use_cached_partitions=True,
    progress_every_s=5,
    compact_after_ingest=True,
    # Output whitelist options
    output_whitelist_path="default",
    output_whitelist_top_n=30_000,
    output_whitelist_year_range=(1900, 2019),
    output_whitelist_spell_check=True,
    output_whitelist_spell_check_language="en_US"
)

build_processed_db(
    filter_config=filter_config,
    pipeline_config=pipeline_config
);


N-GRAM FILTER PIPELINE
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Start Time: 2026-02-07 19:51:37
Mode:       RESUME

Configuration
════════════════════════════════════════════════════════════════════════════════════════════════════
Source DB:            /scratch/edk202/NLP_corpora/Google_Books/20200217/eng/1gram_files/1grams.db
Target DB:            ...dk202/NLP_corpora/Google_Books/20200217/eng/1gram_files/1grams_processed.db
Temp directory:       ...tch/edk202/NLP_corpora/Google_Books/20200217/eng/1gram_files/processing_tmp

Parallelism
────────────────────────────────────────────────────────────────────────────────────────────────────
Workers:              20
Initial work units:   300

Database Profiles
────────────────────────────────────────────────────────────────────────────────────────────────────
Reader profile:       read:packed24
Writer profile:       write:packed24

Ingestion Configuration
─────────────────────────

## **Optional: Inspect Database Files**

### `db_head`: Show first N records

In [7]:
db = f'{db_path_stub}{release}/{language}/{ngram_size}gram_files/{ngram_size}grams_processed.db'

db_head(db, n=5)

First 5 key-value pairs:
────────────────────────────────────────────────────────────────────────────────────────────────────
[ 1] Key:   FALSE
     Value: Total: 124,503 occurrences in 110,465 volumes (1538-2019, 380 bins)

[ 2] Key:   TRUE
     Value: Total: 4,241,065 occurrences in 2,916,658 volumes (1501-2019, 440 bins)

[ 3] Key:   aaa
     Value: Total: 4,449,225 occurrences in 1,052,377 volumes (1477-2019, 402 bins)

[ 4] Key:   aaaa
     Value: Total: 472,912 occurrences in 91,764 volumes (1477-2019, 337 bins)

[ 5] Key:   aaaaa
     Value: Total: 54,371 occurrences in 22,966 volumes (1581-2019, 274 bins)



### `db_peek`: Show records starting from a key

In [8]:
db = f'{db_path_stub}{release}/{language}/{ngram_size}gram_files/{ngram_size}grams_processed.db'

db_peek(db, start_key="police", n=5)

5 key-value pairs starting from 706f6c696365:
────────────────────────────────────────────────────────────────────────────────────────────────────
[ 1] Key:   police
     Value: Total: 166,737,682 occurrences in 13,772,271 volumes (1478-2019, 410 bins)

[ 2] Key:   policea
     Value: Total: 403 occurrences in 309 volumes (1804-2019, 125 bins)

[ 3] Key:   policeaan
     Value: Total: 139 occurrences in 63 volumes (1860-1996, 32 bins)

[ 4] Key:   policeability
     Value: Total: 269 occurrences in 125 volumes (1963-2019, 24 bins)

[ 5] Key:   policeable
     Value: Total: 1,433 occurrences in 1,147 volumes (1865-2019, 77 bins)



### `db_peek_prefix`: Show records matching a prefix

In [9]:
db = f'{db_path_stub}{release}/{language}/{ngram_size}gram_files/{ngram_size}grams_processed.db'

db_peek_prefix(db, prefix="doctor", n=5)

5 key-value pairs with prefix 646f63746f72:
────────────────────────────────────────────────────────────────────────────────────────────────────
[ 1] Key:   doctor
     Value: Total: 127,487,528 occurrences in 18,226,345 volumes (1476-2019, 486 bins)

[ 2] Key:   doctora
     Value: Total: 22,165 occurrences in 11,203 volumes (1644-2019, 215 bins)

[ 3] Key:   doctoraal
     Value: Total: 5,332 occurrences in 2,015 volumes (1878-2019, 94 bins)

[ 4] Key:   doctoraalexamen
     Value: Total: 441 occurrences in 190 volumes (1919-2018, 56 bins)

[ 5] Key:   doctoraalscriptie
     Value: Total: 934 occurrences in 674 volumes (1961-2019, 53 bins)

